# 04.3 — Length-Controlled Analysis

Isolates the length-controlled robustness checks first developed in `04.2_baseline_comparison.ipynb`, once comment length (`word_count`) turned out to correlate strongly with `is_convincing` (r=0.37) and to out-predict both TF-IDF and the engineered-feature pipeline on its own. That notebook stays focused on comparing baseline architectures; this one asks a different question: how much of any model's apparent skill survives once comment length stops separating the classes.

Self-contained, like the other `04.x` notebooks: reloads the data and rebuilds the features, split, and reference-point models independently rather than depending on `04.1_model_training.ipynb` or `04.2_baseline_comparison.ipynb` having been run first.

## Setup

Reproduces the data load, `thread_id` grouping, feature matrix, and thread-grouped train/test split from `04.1_model_training.ipynb` (sections 8.1-8.3) and `04.2_baseline_comparison.ipynb` — see those notebooks for the reasoning behind each step.

In [1]:
import pandas as pd
import zipfile
import json

with zipfile.ZipFile('../results/03_cmv_comments_df.csv.zip') as z:
    comments_df = pd.read_csv(z.open(z.namelist()[0]))
comments_df = comments_df.copy()  # consolidate blocks - avoids a pandas PerformanceWarning on later single-column inserts

with open('../results/03_untrusted_features.json') as f:
    untrusted_features = json.load(f)

print(f'Loaded {len(comments_df)} rows, {comments_df.shape[1]} columns')
print(f'Untrusted features (used below only for the selected-pipeline reference point): {untrusted_features}')

Loaded 3971 rows, 237 columns
Untrusted features (used below only for the selected-pipeline reference point): ['sentiment_vader', 'tone_label', 'style_label', 'ethos_score', 'pathos_score', 'logos_score']


In [2]:
import hashlib

comments_df['thread_id'] = comments_df['original_post'].apply(
    lambda x: hashlib.md5(str(x).encode()).hexdigest()
)

print(f'Unique threads : {comments_df["thread_id"].nunique()}')
print(f'Total comments : {len(comments_df)}')

Unique threads : 401
Total comments : 3971


In [3]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder

# Comment-side, OP-side, and comment↔OP diff features from 03.1-03.3.
numeric_features = [
    'sentiment_llm', 'adj_adv_ratio', 'fk_grade_level', 'fk_reading_ease', 'evidence_count',
    'ethos_llm', 'pathos_llm', 'logos_llm',
    'sentiment_llm_op', 'adj_adv_ratio_op', 'fk_grade_level_op', 'evidence_count_op',
    'ethos_llm_op', 'pathos_llm_op', 'logos_llm_op',
    'sentiment_diff', 'ethos_diff', 'pathos_diff', 'logos_diff',
    'adj_adv_ratio_diff', 'fk_grade_level_diff', 'evidence_count_diff',
]
embedding_features = [c for c in comments_df.columns if c.startswith('embedding_')]
label = 'is_convincing'

categorical_features = ['tone_google', 'tone_google_op']
onehot_encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
tone_encoded = onehot_encoder.fit_transform(comments_df[categorical_features].fillna('unknown'))
tone_df = pd.DataFrame(tone_encoded, columns=onehot_encoder.get_feature_names_out(), index=comments_df.index)

comments_df['is_formal']    = (comments_df['style_google']    == 'Formal').astype(int)
comments_df['is_formal_op'] = (comments_df['style_google_op'] == 'Formal').astype(int)

match_features = ['sentiment_match', 'tone_match', 'style_match', 'ethos_match', 'pathos_match', 'logos_match']
match_df = comments_df[match_features].astype(int)

X = pd.concat([tone_df,
               comments_df[['is_formal', 'is_formal_op', 'use_of_persuasive_lang'] + numeric_features],
               match_df,
               comments_df[embedding_features]],
              axis=1)
y = comments_df[label]
groups = comments_df['thread_id']

print(f'Features : {X.shape[1]}  ({len(embedding_features)} of them PCA embedding components)')
print(f'Samples  : {X.shape[0]}')

Features : 228  (187 of them PCA embedding components)
Samples  : 3971


In [4]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_threads = set(groups.iloc[train_idx])
test_threads  = set(groups.iloc[test_idx])
assert len(train_threads & test_threads) == 0, 'LEAKAGE: shared threads between train and test!'
print(f'Train: {len(train_idx)} comments from {len(train_threads)} threads')
print(f'Test : {len(test_idx)} comments from {len(test_threads)} threads')

Train: 3262 comments from 320 threads
Test : 709 comments from 81 threads


## Reference Points

Three reference points from `04.2_baseline_comparison.ipynb` are needed throughout this notebook: the selected engineered-feature pipeline (XGBoost + SMOTE on `trusted+untrusted`), `TF-IDF + LogReg`, and `word_count` alone. All three are refit here, on the same thread-grouped split (same seed), rather than depending on that notebook having been run first.

In [5]:
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
import warnings
warnings.filterwarnings('ignore')

untrusted_encoded = []
for col in untrusted_features:
    if pd.api.types.is_numeric_dtype(comments_df[col]):
        untrusted_encoded.append(comments_df[[col]].fillna(0))
    else:
        ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
        encoded = ohe.fit_transform(comments_df[[col]].fillna('unknown'))
        untrusted_encoded.append(pd.DataFrame(encoded, columns=ohe.get_feature_names_out(), index=comments_df.index))

X_with_untrusted = pd.concat([X] + untrusted_encoded, axis=1)
X_train_selected = X_with_untrusted.iloc[train_idx]
X_test_selected  = X_with_untrusted.iloc[test_idx]

selected_pipe = Pipeline([
    ('scaler', RobustScaler()),
    ('sampler', SMOTE(random_state=42)),
    ('clf', xgb.XGBClassifier(random_state=42, eval_metric='logloss')),
])
selected_pipe.fit(X_train_selected, y_train)
proba_selected = selected_pipe.predict_proba(X_test_selected)[:, 1]
pred_selected  = selected_pipe.predict(X_test_selected)

pr_auc_selected  = average_precision_score(y_test, proba_selected)
roc_auc_selected = roc_auc_score(y_test, proba_selected)
f1_selected       = f1_score(y_test, pred_selected, zero_division=0)

print('Selected pipeline (XGB + SMOTE oversampling, trusted+untrusted features):')
print(f'Test PR-AUC  : {pr_auc_selected:.4f}')
print(f'Test ROC-AUC : {roc_auc_selected:.4f}')
print(f'Test F1 @ 0.5: {f1_selected:.4f}  (untuned threshold - see 04.1_model_training.ipynb for the validation-tuned figure)')

Selected pipeline (XGB + SMOTE oversampling, trusted+untrusted features):
Test PR-AUC  : 0.3848
Test ROC-AUC : 0.6982
Test F1 @ 0.5: 0.2513  (untuned threshold - see 04.1_model_training.ipynb for the validation-tuned figure)


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

text_train = comments_df['cleaned_final_comment'].iloc[train_idx].fillna('')
text_test  = comments_df['cleaned_final_comment'].iloc[test_idx].fillna('')

tfidf = TfidfVectorizer(max_features=5000, min_df=2)
X_text_train = tfidf.fit_transform(text_train)
X_text_test  = tfidf.transform(text_test)

tfidf_logreg = LogisticRegression(random_state=42, max_iter=1000)
tfidf_logreg.fit(X_text_train, y_train)
proba = tfidf_logreg.predict_proba(X_text_test)[:, 1]
pred  = tfidf_logreg.predict(X_text_test)

tfidf_alone = {
    'PR-AUC':  average_precision_score(y_test, proba),
    'ROC-AUC': roc_auc_score(y_test, proba),
    'F1':      f1_score(y_test, pred, zero_division=0),
}

reference_points = pd.DataFrame({
    'Selected pipeline (XGB + SMOTE, trusted+untrusted)': {'PR-AUC': pr_auc_selected, 'ROC-AUC': roc_auc_selected, 'F1': f1_selected},
    'TF-IDF + LogReg': tfidf_alone,
}).T.round(4)
reference_points

,PR-AUC,ROC-AUC,F1
"Selected pipeline (XGB + SMOTE, trusted+untrusted)",0.3848,0.6982,0.2513
TF-IDF + LogReg,0.5119,0.8150,0.1235


### `word_count` as a Reference Point

`04.2_baseline_comparison.ipynb` traces how the checks below got motivated: several of `TF-IDF + LogReg`'s strongest coefficients turned out to be plain stopwords, `word_count` alone (recomputed below) turned out to out-predict both TF-IDF and the full engineered pipeline on the honest thread-grouped split, and two within-thread checks (fixed-effects correlation r=0.29 vs. r=0.37 across the whole dataset; the convincing comment is longer than its own thread's non-convincing average in 76.5% of paired threads) confirmed the effect isn't just a hot-topics/thread-size artifact. `word_count` is recomputed here as this notebook's third reference point.

In [7]:
from scipy.stats import pointbiserialr

# word_count on the same cleaned_final_comment text TF-IDF is built from, so it's measuring
# length the way TF-IDF actually "sees" it (URLs/punctuation already stripped upstream).
comments_df['word_count'] = comments_df['cleaned_final_comment'].fillna('').apply(lambda s: len(s.split()))

print(comments_df['word_count'].describe())

r, p = pointbiserialr(comments_df['is_convincing'], comments_df['word_count'])
print(f'\nPoint-biserial correlation (word_count vs is_convincing): r={r:.4f}, p={p:.2e}')

mean_by_class = comments_df.groupby('is_convincing')['word_count'].mean()
print(f'Mean word count - not convincing: {mean_by_class[0]:.1f}, convincing: {mean_by_class[1]:.1f}')

count    3971.000000
mean      100.403425
std       137.338553
min         0.000000
25%        22.000000
50%        56.000000
75%       122.500000
max      1573.000000
Name: word_count, dtype: float64

Point-biserial correlation (word_count vs is_convincing): r=0.3716, p=2.92e-130
Mean word count - not convincing: 78.0, convincing: 216.7


In [8]:
# word_count alone, evaluated the same honest way as every other baseline in this notebook -
# thread-grouped train/test split, scaled (single continuous feature, same convention as the
# rest of the notebook's scaled variants).
word_count_train = comments_df['word_count'].iloc[train_idx].values.reshape(-1, 1)
word_count_test  = comments_df['word_count'].iloc[test_idx].values.reshape(-1, 1)

wc_scaler = RobustScaler()
word_count_train_scaled = wc_scaler.fit_transform(word_count_train)
word_count_test_scaled  = wc_scaler.transform(word_count_test)

wc_logreg = LogisticRegression(random_state=42, max_iter=1000)
wc_logreg.fit(word_count_train_scaled, y_train)
proba_wc = wc_logreg.predict_proba(word_count_test_scaled)[:, 1]
pred_wc  = wc_logreg.predict(word_count_test_scaled)

word_count_result = {
    'PR-AUC':  average_precision_score(y_test, proba_wc),
    'ROC-AUC': roc_auc_score(y_test, proba_wc),
    'F1':      f1_score(y_test, pred_wc, zero_division=0),
}

length_vs_others = pd.DataFrame({
    'word_count alone':             word_count_result,
    'TF-IDF alone':                 tfidf_alone,
    'Selected engineered pipeline': {'PR-AUC': pr_auc_selected, 'ROC-AUC': roc_auc_selected, 'F1': f1_selected},
}).T.round(4)
length_vs_others

,PR-AUC,ROC-AUC,F1
word_count alone,0.5256,0.8294,0.2105
TF-IDF alone,0.5119,0.8150,0.1235
Selected engineered pipeline,0.3848,0.6982,0.2513


## 8.11 Robustness Check: Length-Matched Resampling

`04.2_baseline_comparison.ipynb`'s within-thread checks show the length↔convincing relationship isn't purely a hot-topics artifact — it holds even comparing comments within the same thread (fixed-effects correlation r=0.29; the convincing comment is longer than its own thread's non-convincing average in 76.5% of paired threads). This section asks the complementary question directly, via resampling rather than statistical control: if non-convincing comments are selected to have a length distribution closer to convincing ones, does anything else in the reference-point comparisons above change?

Two variants are built from the existing dataset — no need to re-run `01`-`03.3`, every feature is already computed for every comment, this is just a different row selection before modeling:

- **Loose length floor**: keep every convincing comment, but restrict non-convincing comments to at least the convincing group's 25th percentile length (84 words) — removes the shortest "easy negative" comments while keeping most of the non-convincing pool and its natural class imbalance.
- **Strict 1:1 match**: bin convincing comments' lengths into deciles and randomly sample non-convincing comments from the same bins in matching counts — forces the two classes' length distributions to be near-identical (not just their means), with balanced classes as a side effect.

Each variant gets a fresh thread-grouped train/test split (row composition changed, so the original split can't be reused) and is evaluated the same way as the reference points above: `TF-IDF + LogReg`, `word_count` alone, and the selected engineered pipeline (XGB + SMOTE on `trusted+untrusted`). These are single-split point estimates on notably smaller samples than the full dataset, so read differences with the same caution as everywhere else in this notebook that reports only a point estimate.

In [9]:
# Loose length floor: keep every convincing comment, restrict non-convincing comments to at
# least the convincing group's 25th percentile length (84 words) - removes the very-short
# "easy negative" comments driving the mean/median gap, while keeping most of the pool.
conv_mask = comments_df['is_convincing'] == 1
loose_floor = comments_df.loc[conv_mask, 'word_count'].quantile(0.25)
loose_mask = conv_mask | (comments_df['word_count'] >= loose_floor)
comments_df_loose = comments_df[loose_mask].copy()

print(f'Loose length-floor sample (floor={loose_floor:.0f} words):')
print(f'  {len(comments_df_loose)} comments ({int(comments_df_loose["is_convincing"].sum())} convincing, '
      f'positive rate {comments_df_loose["is_convincing"].mean():.3f}) - vs. {len(comments_df)} / '
      f'{int(comments_df["is_convincing"].sum())} / {comments_df["is_convincing"].mean():.3f} in the full dataset')
r_loose, p_loose = pointbiserialr(comments_df_loose['is_convincing'], comments_df_loose['word_count'])
print(f'  word_count correlation after filtering: r={r_loose:.4f}, p={p_loose:.2e}  (full dataset: r={r:.4f})')
print()

# Strict 1:1 match: bin convincing comments' word_count into deciles, then randomly sample
# non-convincing comments from the same bins in matching counts - forces the two classes'
# length distributions to be near-identical, not just their means, and balances the classes
# as a side effect.
n_bins = 10
conv_df    = comments_df[comments_df['is_convincing'] == 1].copy()
nonconv_df = comments_df[comments_df['is_convincing'] == 0].copy()

bin_edges = pd.qcut(conv_df['word_count'], n_bins, duplicates='drop', retbins=True)[1]
conv_df['length_bin']    = pd.cut(conv_df['word_count'], bins=bin_edges, include_lowest=True)
nonconv_df['length_bin'] = pd.cut(nonconv_df['word_count'], bins=bin_edges, include_lowest=True)

matched_parts = []
for bin_label, group in conv_df.groupby('length_bin', observed=True):
    pool = nonconv_df[nonconv_df['length_bin'] == bin_label]
    take_n = min(len(group), len(pool))
    if take_n > 0:
        matched_parts.append(pool.sample(n=take_n, random_state=42))

matched_nonconv_df = pd.concat(matched_parts) if matched_parts else nonconv_df.iloc[:0]
comments_df_matched = pd.concat([conv_df, matched_nonconv_df]).drop(columns='length_bin').sort_index()

print(f'Strict 1:1 length-matched sample:')
print(f'  {len(comments_df_matched)} comments ({int(comments_df_matched["is_convincing"].sum())} convincing, '
      f'positive rate {comments_df_matched["is_convincing"].mean():.3f})')
r_matched, p_matched = pointbiserialr(comments_df_matched['is_convincing'], comments_df_matched['word_count'])
print(f'  word_count correlation after matching: r={r_matched:.4f}, p={p_matched:.2e}  (full dataset: r={r:.4f})')

Loose length-floor sample (floor=84 words):
  1638 comments (641 convincing, positive rate 0.391) - vs. 3971 / 641 / 0.161 in the full dataset
  word_count correlation after filtering: r=0.0930, p=1.63e-04  (full dataset: r=0.3716)

Strict 1:1 length-matched sample:
  1235 comments (641 convincing, positive rate 0.519)
  word_count correlation after matching: r=0.0655, p=2.14e-02  (full dataset: r=0.3716)


In [10]:
def evaluate_length_sample(sample_df, seed=42):
    """Rebuild a thread-grouped train/test split, TF-IDF+LR, word_count alone, and the
    selected engineered pipeline (XGB+SMOTE on trusted+untrusted) on a length-filtered
    subsample of comments_df - reusing already-computed columns (X_with_untrusted, word_count)
    rather than recomputing them."""
    idx = sample_df.index
    y_sample = comments_df.loc[idx, 'is_convincing']
    groups_sample = comments_df.loc[idx, 'thread_id']

    gss_sample = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_pos, te_pos = next(gss_sample.split(np.zeros((len(idx), 1)), y_sample, groups=groups_sample))
    tr_idx, te_idx = idx[tr_pos], idx[te_pos]

    y_tr, y_te = y_sample.loc[tr_idx], y_sample.loc[te_idx]

    # TF-IDF + LogReg
    text_tr = comments_df.loc[tr_idx, 'cleaned_final_comment'].fillna('')
    text_te = comments_df.loc[te_idx, 'cleaned_final_comment'].fillna('')
    tfidf_s = TfidfVectorizer(max_features=5000, min_df=2)
    Xtr_text = tfidf_s.fit_transform(text_tr)
    Xte_text = tfidf_s.transform(text_te)
    lr_s = LogisticRegression(random_state=42, max_iter=1000)
    lr_s.fit(Xtr_text, y_tr)
    proba_tfidf = lr_s.predict_proba(Xte_text)[:, 1]

    # word_count alone
    wc_tr = comments_df.loc[tr_idx, ['word_count']].values
    wc_te = comments_df.loc[te_idx, ['word_count']].values
    wc_scaler_s = RobustScaler()
    wc_tr_s = wc_scaler_s.fit_transform(wc_tr)
    wc_te_s = wc_scaler_s.transform(wc_te)
    wc_lr_s = LogisticRegression(random_state=42, max_iter=1000)
    wc_lr_s.fit(wc_tr_s, y_tr)
    proba_wc = wc_lr_s.predict_proba(wc_te_s)[:, 1]

    # Selected engineered pipeline architecture (XGB + SMOTE, trusted+untrusted) - reuses
    # X_with_untrusted (already built in the Reference Point section) rather than re-encoding.
    Xeng_tr = X_with_untrusted.loc[tr_idx]
    Xeng_te = X_with_untrusted.loc[te_idx]
    eng_pipe_s = Pipeline([
        ('scaler', RobustScaler()),
        ('sampler', SMOTE(random_state=42)),
        ('clf', xgb.XGBClassifier(random_state=42, eval_metric='logloss')),
    ])
    eng_pipe_s.fit(Xeng_tr, y_tr)
    proba_eng = eng_pipe_s.predict_proba(Xeng_te)[:, 1]

    return {
        'n_total': len(idx), 'n_train': len(tr_idx), 'n_test': len(te_idx),
        'positive_rate': float(y_sample.mean()),
        # The exact test-fold positive rate - not the whole-sample rate above - is the correct
        # no-skill PR-AUC baseline for this specific split. Thread-grouped splits on a small
        # sample can drift noticeably from the sample-wide rate (see 04.1_model_training.ipynb
        # section 8.5 for why PR-AUC's no-skill baseline is the positive rate in the first place).
        'test_positive_rate': float(y_te.mean()),
        'TF-IDF + LogReg':     {'PR-AUC': average_precision_score(y_te, proba_tfidf), 'ROC-AUC': roc_auc_score(y_te, proba_tfidf)},
        'word_count alone':    {'PR-AUC': average_precision_score(y_te, proba_wc),    'ROC-AUC': roc_auc_score(y_te, proba_wc)},
        'Engineered pipeline': {'PR-AUC': average_precision_score(y_te, proba_eng),   'ROC-AUC': roc_auc_score(y_te, proba_eng)},
    }

In [11]:
results_loose   = evaluate_length_sample(comments_df_loose)
results_matched = evaluate_length_sample(comments_df_matched)

# The exact test-fold positive rate (not the whole-sample rate) is the correct no-skill PR-AUC
# baseline for each row - thread-grouped test folds on small samples can drift from the
# sample-wide rate, as the model re-selection check below (§8.12) makes concrete: the strict-
# match test fold's positive rate turns out to be 0.468, not the whole-sample's 0.519.
summary_rows = []
for sample_name, res, test_pos_rate in [
    ('Full dataset',        None,            float(y_test.mean())),
    ('Loose length floor',  results_loose,   results_loose['test_positive_rate']),
    ('Strict 1:1 match',    results_matched, results_matched['test_positive_rate']),
]:
    if res is None:
        row = {
            'n': len(comments_df), 'positive rate': comments_df['is_convincing'].mean(),
            'test positive rate': test_pos_rate,
            'TF-IDF PR-AUC': tfidf_alone['PR-AUC'],
            'word_count PR-AUC': word_count_result['PR-AUC'],
            'Engineered PR-AUC': pr_auc_selected,
        }
    else:
        row = {
            'n': res['n_total'], 'positive rate': res['positive_rate'],
            'test positive rate': test_pos_rate,
            'TF-IDF PR-AUC': res['TF-IDF + LogReg']['PR-AUC'],
            'word_count PR-AUC': res['word_count alone']['PR-AUC'],
            'Engineered PR-AUC': res['Engineered pipeline']['PR-AUC'],
        }
    row['sample'] = sample_name
    summary_rows.append(row)

length_robustness_summary = pd.DataFrame(summary_rows).set_index('sample').round(4)

# Lift over no-skill = PR-AUC minus THIS ROW'S OWN test-fold positive rate (see note above) -
# the fair, sample-independent comparison, and the correct one (not the whole-sample rate
# used in an earlier version of this table).
for col in ['TF-IDF PR-AUC', 'word_count PR-AUC', 'Engineered PR-AUC']:
    lift_col = col.replace(' PR-AUC', ' lift over no-skill')
    length_robustness_summary[lift_col] = (length_robustness_summary[col] - length_robustness_summary['test positive rate']).round(4)

length_robustness_summary

,n,positive rate,test positive rate,TF-IDF PR-AUC,word_count PR-AUC,Engineered PR-AUC,TF-IDF lift over no-skill,word_count lift over no-skill,Engineered lift over no-skill
sample,,,,,,,,,
Full dataset,3971,0.1614,0.2031,0.5119,0.5256,0.3848,0.3088,0.3225,0.1817
Loose length floor,1638,0.3913,0.3903,0.4381,0.4801,0.4068,0.0478,0.0898,0.0165
Strict 1:1 match,1235,0.5190,0.4679,0.5446,0.5255,0.5013,0.0767,0.0576,0.0334


**Correction: an earlier version of this analysis used the whole-sample positive rate as the no-skill baseline for the lift calculation. The correct baseline is each row's own *test-fold* positive rate**, which can drift noticeably from the sample-wide rate on a thread-grouped split this small — the strict-match test fold's positive rate is 0.468, not the whole-sample's 0.519 (and the full dataset's test fold is 0.203, not 0.161 — this matches the `Majority-class` baseline's PR-AUC of 0.2031 already reported in §8.7, which is exactly the test-fold positive rate by construction, a useful consistency check that this correction is right). The table above and the discussion below use the corrected `test positive rate` column throughout.

**Every model's lift over no-skill still shrinks sharply once length stops separating the classes — but none of them go negative once the baseline is computed correctly.** TF-IDF: 0.309 → 0.048 → 0.077. `word_count`: 0.323 → 0.090 → 0.058 (as expected, this is the one designed to collapse, and it does — though not all the way to zero, since decile-bin matching leaves a small residual correlation, r=0.093 loose / r=0.065 strict, as reported earlier). The engineered pipeline: 0.182 → 0.017 → **0.033** — modestly positive in the strict match, not below no-skill as an earlier version of this section reported. That earlier claim was an artifact of the imprecise baseline, not a real finding — worth flagging plainly rather than quietly fixing, since it changes the headline conclusion.

**Revised bottom line**: length-matching still causes a dramatic collapse in every model's real skill (roughly 75-90% of each model's lift disappears between the full dataset and the strict match) — that part of the earlier conclusion holds. But "the engineered pipeline has *no* signal left, or negative signal" was wrong. All three approaches retain a small, comparable, positive lift in the strict-matched sample (TF-IDF 0.077, `word_count` 0.058, engineered 0.033) — modest, and on a sample this small (1,235 rows, ~265 test rows) not something to treat as a confidently-ranked comparison between them, but not zero or negative either. The honest statement is: once length is controlled, there's a little real signal left in each approach, not none.

## 8.12 Re-selecting the Best Model on the Length-Matched Sample

The engineered pipeline's collapse to below-no-skill in the strict 1:1 match (§8.11) reused the exact architecture selected in `04.1_model_training.ipynb`'s validation grid — XGBoost + SMOTE oversampling on `trusted+untrusted` — which was chosen on the *original*, length-imbalanced full dataset. That choice might not transfer: a much smaller (1,235 rows), already-balanced (52% positive) sample doesn't need oversampling, and a complex model like XGBoost can overfit more easily with less data. This reruns the same validation-grid methodology (9 classifiers × 3 resampling strategies × 3 feature sets) from scratch on `comments_df_matched`, to check whether a different combination finds genuine skill, or whether every combination still lands near no-skill regardless of architecture.

In [12]:
# Same thread-grouped 80/20 split logic as evaluate_length_sample() above, computed explicitly
# here (rather than reusing that function) so the split, inner train/val split, and the full
# grid below are all visible and directly comparable to 04.1_model_training.ipynb's approach.
matched_idx     = comments_df_matched.index
y_matched       = comments_df.loc[matched_idx, 'is_convincing']
groups_matched  = comments_df.loc[matched_idx, 'thread_id']

gss_matched = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_pos, te_pos = next(gss_matched.split(np.zeros((len(matched_idx), 1)), y_matched, groups=groups_matched))
train_idx_matched, test_idx_matched = matched_idx[tr_pos], matched_idx[te_pos]

y_train_matched = y_matched.loc[train_idx_matched]
y_test_matched  = y_matched.loc[test_idx_matched]

feature_sets_matched = {
    'trusted':           X.loc[matched_idx],
    'trusted+untrusted': X_with_untrusted.loc[matched_idx],
    'embeddings-only':   X.loc[matched_idx, embedding_features],
}
print({name: f'{fs.shape[1]} features' for name, fs in feature_sets_matched.items()})
print(f'Train: {len(train_idx_matched)} comments from {groups_matched.loc[train_idx_matched].nunique()} threads')
print(f'Test : {len(test_idx_matched)} comments from {groups_matched.loc[test_idx_matched].nunique()} threads')

{'trusted': '228 features', 'trusted+untrusted': '239 features', 'embeddings-only': '187 features'}
Train: 970 comments from 311 threads
Test : 265 comments from 78 threads


In [13]:
from sklearn.ensemble import RandomForestClassifier
from imblearn.under_sampling import RandomUnderSampler

def make_pipelines_matched(clf, resampling='none'):
    steps = [('scaler', RobustScaler())]
    if resampling == 'under':
        steps.append(('sampler', RandomUnderSampler(random_state=42)))
    elif resampling == 'over':
        steps.append(('sampler', SMOTE(random_state=42)))
    steps.append(('clf', clf))
    return Pipeline(steps)

pipelines_matched = {
    'LogReg (none)':  make_pipelines_matched(LogisticRegression(random_state=42, max_iter=1000), 'none'),
    'LogReg (under)': make_pipelines_matched(LogisticRegression(random_state=42, max_iter=1000), 'under'),
    'LogReg (over)':  make_pipelines_matched(LogisticRegression(random_state=42, max_iter=1000), 'over'),
    'RF (none)':      make_pipelines_matched(RandomForestClassifier(random_state=42, n_estimators=100), 'none'),
    'RF (under)':     make_pipelines_matched(RandomForestClassifier(random_state=42, n_estimators=100), 'under'),
    'RF (over)':      make_pipelines_matched(RandomForestClassifier(random_state=42, n_estimators=100), 'over'),
    'XGB (none)':     make_pipelines_matched(xgb.XGBClassifier(random_state=42, eval_metric='logloss'), 'none'),
    'XGB (under)':    make_pipelines_matched(xgb.XGBClassifier(random_state=42, eval_metric='logloss'), 'under'),
    'XGB (over)':     make_pipelines_matched(xgb.XGBClassifier(random_state=42, eval_metric='logloss'), 'over'),
}
print(f'{len(pipelines_matched)} pipelines defined')

9 pipelines defined


In [14]:
# Thread-disjoint inner-train/validation split, carved from train_idx_matched - mirrors
# 04.1_model_training.ipynb's approach (same random_state=0) so model selection never touches
# the held-out test set.
train_idx_matched_arr = np.array(train_idx_matched)
train_groups_matched  = groups_matched.loc[train_idx_matched].reset_index(drop=True)
y_train_matched_r     = y_train_matched.reset_index(drop=True)

gss_val_matched = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
inner_pos, val_pos = next(gss_val_matched.split(np.zeros((len(train_idx_matched), 1)), y_train_matched_r, groups=train_groups_matched))

inner_idx_matched = train_idx_matched_arr[inner_pos]
val_idx_matched   = train_idx_matched_arr[val_pos]

y_inner_matched = y_matched.loc[inner_idx_matched]
y_val_matched   = y_matched.loc[val_idx_matched]

inner_threads_matched = set(groups_matched.loc[inner_idx_matched])
val_threads_matched   = set(groups_matched.loc[val_idx_matched])
assert len(inner_threads_matched & val_threads_matched) == 0
print(f'Inner train     : {len(inner_idx_matched)} comments from {len(inner_threads_matched)} threads')
print(f'Validation      : {len(val_idx_matched)} comments from {len(val_threads_matched)} threads')
print(f'Test (held out) : {len(test_idx_matched)} comments from {groups_matched.loc[test_idx_matched].nunique()} threads')

Inner train     : 765 comments from 248 threads
Validation      : 205 comments from 63 threads
Test (held out) : 265 comments from 78 threads


In [15]:
val_metrics_matched  = {}
combo_lookup_matched = {}

for fs_name, X_variant in feature_sets_matched.items():
    X_inner_variant = X_variant.loc[inner_idx_matched]
    X_val_variant   = X_variant.loc[val_idx_matched]

    for model_name, pipe in pipelines_matched.items():
        combo_name = f'{model_name} | {fs_name}'
        pipe.fit(X_inner_variant, y_inner_matched)
        proba_val = pipe.predict_proba(X_val_variant)[:, 1]

        val_metrics_matched[combo_name] = {
            'PR-AUC':  round(average_precision_score(y_val_matched, proba_val), 4),
            'ROC-AUC': round(roc_auc_score(y_val_matched, proba_val), 4),
        }
        combo_lookup_matched[combo_name] = (model_name, fs_name)

val_df_matched = pd.DataFrame(val_metrics_matched).T.sort_values('PR-AUC', ascending=False)
print(f'Validation results across {len(val_metrics_matched)} combinations on the length-matched sample')
print(f'(validation positive rate: {y_val_matched.mean():.3f} - the no-skill PR-AUC reference for this table):')
val_df_matched.head(10)

Validation results across 27 combinations on the length-matched sample
(validation positive rate: 0.551 - the no-skill PR-AUC reference for this table):


,PR-AUC,ROC-AUC
RF (none) | embeddings-only,0.6656,0.6054
RF (over) | trusted,0.5995,0.5473
XGB (over) | trusted,0.5994,0.5687
RF (under) | trusted,0.5812,0.5313
XGB (under) | trusted,0.5805,0.5325
RF (none) | trusted,0.5775,0.5171
XGB (over) | trusted+untrusted,0.5707,0.4933
RF (over) | embeddings-only,0.5678,0.5213
XGB (none) | trusted,0.5669,0.5297
RF (none) | trusted+untrusted,0.5656,0.4989


In [16]:
best_combo_matched = val_df_matched.index[0]
best_model_name_matched, best_fs_name_matched = combo_lookup_matched[best_combo_matched]
best_pipe_matched   = pipelines_matched[best_model_name_matched]
X_selected_matched  = feature_sets_matched[best_fs_name_matched]

best_pipe_matched.fit(X_selected_matched.loc[train_idx_matched], y_train_matched)
proba_test_matched = best_pipe_matched.predict_proba(X_selected_matched.loc[test_idx_matched])[:, 1]

pr_auc_test_matched         = average_precision_score(y_test_matched, proba_test_matched)
roc_auc_test_matched        = roc_auc_score(y_test_matched, proba_test_matched)
positive_rate_test_matched  = y_test_matched.mean()
lift_reselected             = pr_auc_test_matched - positive_rate_test_matched

# The original (§8.11) run reused the full-dataset-selected XGB+SMOTE/trusted+untrusted
# pipeline on this exact same test set (same GroupShuffleSplit, same seed) - so this is a
# direct, apples-to-apples comparison, now against the *exact* test-fold positive rate
# rather than the whole-sample approximation used in §8.11's table.
original_pr_auc  = results_matched['Engineered pipeline']['PR-AUC']
lift_original    = original_pr_auc - positive_rate_test_matched

print(f'Reselected best combo (by validation PR-AUC): {best_combo_matched}')
print(f'Test PR-AUC        : {pr_auc_test_matched:.4f}')
print(f'Test ROC-AUC        : {roc_auc_test_matched:.4f}')
print(f'Test positive rate  : {positive_rate_test_matched:.4f}')
print(f'Lift over no-skill  : {lift_reselected:.4f}')
print()
print(f'Original pipeline (XGB + SMOTE, trusted+untrusted, selected on the full dataset):')
print(f'Test PR-AUC        : {original_pr_auc:.4f}')
print(f'Lift over no-skill  : {lift_original:.4f}')

Reselected best combo (by validation PR-AUC): RF (none) | embeddings-only
Test PR-AUC        : 0.4518
Test ROC-AUC        : 0.4761
Test positive rate  : 0.4679
Lift over no-skill  : -0.0162

Original pipeline (XGB + SMOTE, trusted+untrusted, selected on the full dataset):
Test PR-AUC        : 0.5013
Lift over no-skill  : 0.0334


**Reselecting made things worse, not better — and that's itself the informative result.** The grid's winner by validation PR-AUC, `RF (none) | embeddings-only` (0.6656 on 205 validation rows — well above that fold's 0.551 no-skill baseline), collapses to PR-AUC 0.4518 on test, *below* the test fold's own no-skill baseline (0.468) — a lift of -0.016. The original architecture (XGB + SMOTE, `trusted+untrusted`, selected on the full 3,971-row dataset and simply reused here) beats it clearly on this same test set: PR-AUC 0.5013, lift +0.033 (per the corrected §8.11 table above).

**This is a textbook small-validation-set failure, not evidence that reselection is pointless in general.** With only 205 validation rows and 27 combinations compared, some combination is likely to look great by chance alone — `RF (none) | embeddings-only`'s 0.6656 was probably exactly that: a validation-set fluke that a fresh grid search dutifully picked up and trusted, because nothing in the selection process can distinguish "genuinely best" from "best due to noise" when the validation set is this small relative to the number of candidates compared.

**Combined conclusion from §8.11 and §8.12**: the original engineered pipeline's modest positive lift (+0.033) on the length-matched sample isn't a fluke of a stale, badly-matched architecture — reselecting from scratch on the same data found nothing better, and in fact found something worse by over-trusting a small validation set. That strengthens confidence in the corrected §8.11 finding: there's a small amount of real, non-length-driven signal in the engineered features, comparable in size to TF-IDF's own residual lift, and not reliably improvable by swapping classifiers or resampling strategies at this sample size. Any future model-selection work on data this small (including Move 2's rhetorical-strategy scores, if evaluated on a similarly small subsample) should budget for this — a bigger validation set, fewer candidates compared, or cross-validation instead of a single validation split would all help avoid the same trap.

## 8.13 A Content-Motivated Length Floor

§8.11's "loose floor" (84 words) was chosen statistically — the convincing group's 25th percentile — not based on what the excluded comments actually contain. A sample of non-convincing comments across length bands, read manually, suggests a different, lower cutoff: below roughly 20-30 words, comments tend to be single-line reactions, affirmations, or off-topic asides; at 30+ words, they consistently contain a complete claim-and-reasoning structure, even when brief. Two bands illustrate the pattern:

In [17]:
for lo, hi, label in [(1, 20, 'Below 20 words (a sample)'), (30, 45, '30-45 words (a sample)')]:
    band = comments_df[(comments_df['is_convincing'] == 0) & (comments_df['word_count'] >= lo) & (comments_df['word_count'] <= hi)]
    sample = band.sample(2, random_state=42)
    print(f'--- {label} ---')
    for _, row in sample.iterrows():
        print(f'[{row["word_count"]} words] {row["final_comment"]}')
        print()

--- Below 20 words (a sample) ---
[17 words] Oh yes I absolutely agree. Sorry we were kinda debating 2 different things here. I wholeheartedly disagree with OP here.

[20 words] Admittedly though that was focused more about handling the birth, but I believe they had things more focused on raising children.

--- 30-45 words (a sample) ---
[33 words] There is no fucking way only 2.7% of people earn minimum wage unless you only count people making the federal minimum wage, which is stupid because most states have their own higher minimum wage.

[34 words] Yeah it’s absolutely crazy to me that you can go into work one day and your employer can decide to just let you go for no reason. How does anyone have any financial stability?



In [18]:
# Content-motivated floor: keep every convincing comment, restrict non-convincing comments to
# at least 30 words - based on the manual reading above, not a statistic of the convincing
# group's own distribution (contrast with the 84-word "loose floor" in §8.11).
content_floor = 30
content_mask = (comments_df['is_convincing'] == 1) | (comments_df['word_count'] >= content_floor)
comments_df_content_floor = comments_df[content_mask].copy()

print(f'Content-motivated floor sample (floor={content_floor} words):')
print(f'  {len(comments_df_content_floor)} comments ({int(comments_df_content_floor["is_convincing"].sum())} convincing, '
      f'positive rate {comments_df_content_floor["is_convincing"].mean():.3f}) - vs. {len(comments_df_loose)} comments '
      f'in the 84-word loose floor, {len(comments_df)} in the full dataset')

r_content, p_content = pointbiserialr(comments_df_content_floor['is_convincing'], comments_df_content_floor['word_count'])
print(f'  word_count correlation after filtering: r={r_content:.4f}, p={p_content:.2e}  '
      f'(full dataset: r={r:.4f}, 84-word loose floor: r={r_loose:.4f})')

Content-motivated floor sample (floor=30 words):
  2753 comments (641 convincing, positive rate 0.233) - vs. 1638 comments in the 84-word loose floor, 3971 in the full dataset
  word_count correlation after filtering: r=0.2882, p=8.70e-54  (full dataset: r=0.3716, 84-word loose floor: r=0.0930)


In [19]:
results_content_floor = evaluate_length_sample(comments_df_content_floor)

content_floor_row = pd.Series({
    'n': results_content_floor['n_total'],
    'positive rate': results_content_floor['positive_rate'],
    'test positive rate': results_content_floor['test_positive_rate'],
    'TF-IDF PR-AUC': results_content_floor['TF-IDF + LogReg']['PR-AUC'],
    'word_count PR-AUC': results_content_floor['word_count alone']['PR-AUC'],
    'Engineered PR-AUC': results_content_floor['Engineered pipeline']['PR-AUC'],
}, name='Content floor (30 words)')

for col in ['TF-IDF PR-AUC', 'word_count PR-AUC', 'Engineered PR-AUC']:
    lift_col = col.replace(' PR-AUC', ' lift over no-skill')
    content_floor_row[lift_col] = content_floor_row[col] - content_floor_row['test positive rate']

content_floor_summary = pd.concat([length_robustness_summary, content_floor_row.to_frame().T])
content_floor_summary.round(4)

,n,positive rate,test positive rate,TF-IDF PR-AUC,word_count PR-AUC,Engineered PR-AUC,TF-IDF lift over no-skill,word_count lift over no-skill,Engineered lift over no-skill
Full dataset,3971.0,0.1614,0.2031,0.5119,0.5256,0.3848,0.3088,0.3225,0.1817
Loose length floor,1638.0,0.3913,0.3903,0.4381,0.4801,0.4068,0.0478,0.0898,0.0165
Strict 1:1 match,1235.0,0.5190,0.4679,0.5446,0.5255,0.5013,0.0767,0.0576,0.0334
Content floor (30 words),2753.0,0.2328,0.2382,0.3905,0.4528,0.2975,0.1523,0.2146,0.0594


**The 30-word content floor keeps far more data than either length-controlled sample (2,753 rows vs. 1,638 loose / 1,235 strict-matched) — but it barely touches the length confound.** The residual `word_count` correlation is r=0.2882 (p=8.7×10⁻⁵⁴) — still 78% of the full dataset's r=0.3716, compared to just 25% remaining after the 84-word floor (r=0.093) or 18% after strict matching (r=0.065). `word_count` alone still carries a lift of 0.215 here — two-thirds of its full-dataset lift (0.323) survives, versus roughly a quarter surviving under either of the more aggressive samples.

**That's the key finding: "remove the useless fragments" and "remove the length confound" are different thresholds, answering different questions, and 30 words only answers the first one.** Manually reading the excluded comments (above) supports the 30-word floor as a *content-quality* cutoff — below it, comments are mostly single-line reactions or off-topic asides with little for a rhetorical-strategy scorer to work with. But removing those specific low-content comments doesn't remove *length as a predictor* from the remaining data, because plenty of length variation (and plenty of the length↔convincing relationship) lives entirely above 30 words too — the 30-50 and 83-word examples earlier in this notebook were all substantive, but still shorter on average than the typical convincing comment.

**Practical implication**: if the goal is a cleaner dataset for Move 2 (dropping comments too short to meaningfully score for rhetorical strategies), a ~30-word floor is well-supported and keeps most of the data. But it should not be treated as a fix for the length confound — anything evaluated on this content-floor sample still needs the same length-aware check used throughout this notebook (or the more aggressive 84-word/matched samples specifically for that purpose), since word_count still explains a large share of what any model finds here.

## 8.14 Boundary Validation: Is 30 Words the Right Cutoff?

§8.13's manual read compared two thin (n=2) samples from far apart length bands — below 20 words and 30-45 words — and found a clear contrast. That's suggestive but doesn't test the cutoff itself: it doesn't say whether comments right around the 30-word line look meaningfully different from each other, only that comments far below it look thinner than comments moderately above it. This reads a larger (n=8 per side), balanced sample drawn tightly around the boundary — 18-29 words vs. 30-41 words — to check the cutoff directly.

In [20]:
below_30 = comments_df[(comments_df['is_convincing'] == 0) & (comments_df['word_count'] >= 18) & (comments_df['word_count'] < 30)]
above_30 = comments_df[(comments_df['is_convincing'] == 0) & (comments_df['word_count'] >= 30) & (comments_df['word_count'] < 42)]
print(f'18-29 words: {len(below_30)} comments available; 30-41 words: {len(above_30)} comments available')

print()
print('=== BELOW 30 (18-29 words) ===')
for _, row in below_30.sample(8, random_state=7).sort_values('word_count').iterrows():
    print(f"--- [{row['word_count']} words] ---")
    print(row['final_comment'])
    print()

print('=== 30 AND ABOVE (30-41 words) ===')
for _, row in above_30.sample(8, random_state=7).sort_values('word_count').iterrows():
    print(f"--- [{row['word_count']} words] ---")
    print(row['final_comment'])
    print()

18-29 words: 479 comments available; 30-41 words: 368 comments available

=== BELOW 30 (18-29 words) ===
--- [18 words] ---
No. They can simply halt trading. Which would screw short sellers over. I don’t see them bailing anyone out.

--- [20 words] ---
It can be a big portion without being super tall or thick. Height is not the only dimension that can increased.

--- [21 words] ---
I'd agree duck meat is definitely tastier, but chickens' white meat is some of the healthiest & protein-rich food you can get.

--- [22 words] ---
I've read about both baby laxatives and meat tenderizers being used to cut heroin! :( (IIRC, "Steal This Urine Test" by Abby Hoffman).

--- [22 words] ---
don't forget that he's also the one who asked John Kerry how he \[Kerry\] got a liberal arts degree in a science (political science)

--- [22 words] ---
Exactly, this is kind of what I was trying to say in my comment too but I don't think I expressed it quite as well.

--- [25 words] ---
I think the other poster

**Most comments even well below 30 words already contain a real claim — the broader §8.13 sample overstated how empty the short side is.** 7 of the 8 below-30 examples (18-29 words) make an actual argumentative point: a claim with a reason ("They can simply halt trading. Which would screw short sellers over."), a comparison ("I'd agree duck meat is definitely tastier, but chickens' white meat is..."), or a cited fact ("I've read about both baby laxatives and meat tenderizers being used to cut heroin... (IIRC...)"). Only one of the eight is a pure content-free affirmation ("Exactly, this is kind of what I was trying to say in my comment too but I don't think I expressed it quite as well") — the pattern §8.13 described as typical below 30 words, but here it's the exception, not the rule.

**The real difference at the boundary is structure, not presence of content.** Below-30 examples are almost always a single compressed clause — one claim, stated once, with no room to develop it further. Above-30 examples (30-41 words) are usually two or more clauses building on each other: a claim *plus* a supporting example, a quoted rebuttal *plus* a clarifying question, or a claim *plus* a closing remark. Both sides argue something; the above-30 side more consistently has room to argue it in more than one move.

**That distinction matters specifically for Move 2's purpose, even though it isn't the "empty vs. substantive" line §8.13 described.** An 8-strategy rubric (justification, concession, evidence, refutation, etc.) needs textual room for more than one strategy to show up in the same comment. A single 18-word clause can be scored, but has little space to exhibit more than one strategy at a time; a 35-word comment with a claim and a supporting example has room for at least two. The 30-word floor still looks like a reasonable, defensible cutoff on those grounds — but it's a threshold for *scoreable structure*, not a threshold between "fragment" and "real argument" the way §8.13 characterized it. Worth keeping that more precise framing in mind if the exact cutoff is ever revisited.